##**Funciones base y generador de lista**

In [4]:
import numpy as np                                                                 # Operaciones numéricas (aunque aquí casi no se usa)
import random                                                                      # Generación de números aleatorios

"""
Nota sobre coordenadas:

Las coordenadas geográficas se almacenan como (lat, lon),
pero matemáticamente corresponden a (y, x).

Por esta razón, en las visualizaciones se trabaja con:
    - latitud  → eje Y
    - longitud → eje X

Esto implica que los cortes y particiones del KD-Tree se realizan
primero sobre Y y luego sobre X, aunque la tupla esté en formato (lat, lon).
"""

data = []

# Límites del Valle de Aburrá
LAT_MIN = 5.98                                                                     # Sur (más allá de Caldas) latitud en grados al norte del ecuador
LAT_MAX = 6.51                                                                     # Norte (Barbosa) latitud en grados al norte del ecuador
LON_MIN = -75.71                                                                   # Occidente, longitud en grados al oeste de Greenwich
LON_MAX = -75.21                                                                   # Oriente, longitud en grados al oeste de Greenwich

def generate_data():
    for i in range(10000):
        lat = random.uniform(LAT_MIN, LAT_MAX)
        lon = random.uniform(LON_MIN, LON_MAX)
        data.append((lat, lon))

generate_data()

In [13]:
import random                                                                      # Generación de números aleatorios
import numpy as np                                                                 # Operaciones numéricas (aunque aquí casi no se usa)

# ------------------------------ Lista ligada ------------------------------------

# Nodo de la lista ligada
class Node:
    def __init__(self, coord):                                                     # Constructor del nodo
        self.coord = coord                                                         # Coordenada almacenada (lat, lon)
        self.next = None                                                           # Referencia al siguiente nodo (enlace)

# Lista ligada
class LinkedList:                                                                  # Estructura de lista simplemente enlazada
    def __init__(self):                                                            # Constructor de la lista
        self.head = None                                                           # Primer nodo de la lista (inicio)
        self.size = 0                                                              # Cantidad de elementos en la lista

    def append(self, coord):                                                       # Inserta un nuevo nodo al final
        new_node = Node(coord)                                                     # Crear nodo con la coordenada

        if not self.head:                                                          # Si la lista está vacía
            self.head = new_node                                                   # El nuevo nodo es el primero
        else:
            current = self.head
            while current.next:                                                    # Recorrer hasta el último nodo
                current = current.next
            current.next = new_node                                                # Enlazar el nuevo nodo al final

        self.size += 1                                                             # Incrementar tamaño de la lista

    def __repr__(self):                                                            # Representación en texto de la lista
        nodes, current = [], self.head
        while current:
            nodes.append(str(current.coord))                                       # Guardar coordenadas como string
            current = current.next
        return " -> ".join(nodes[:5]) + f" ... ({self.size} nodos)"                # Mostrar primeros 5 nodos

# ------------------------------ Lista ligada ------------------------------------




# --------------------------------- Árbol KD -------------------------------------

class Node_KD_2D:                                                                  # Clase para el nodo del árbol
    def __init__(self, value, left=None, right=None):                              # Constructor del nodo
        self.value = value                                                         # Punto almacenado (lat, lon)
        self.left = left                                                           # Subárbol izquierdo (valores menores)
        self.right = right                                                         # Subárbol derecho (valores mayores)

# --------------------------------- Árbol KD -------------------------------------




# ------------------------------- Funciones --------------------------------------

# ------------------------------ Fuerza Bruta ------------------------------------

def F_bruta_list(data):                                                           # Fuerza Bruta para listas
    sorted_by_x = sorted(data, key=lambda coord: coord[1])                        # Ordenar puntos por X (longitud → índice 1)
    sorted_by_y = sorted(data, key=lambda coord: coord[0])                        # Ordenar puntos por Y (latitud → índice 0)

    # Construir lista ligada ordenada por X
    list_x = LinkedList()                                                         # Crear lista ligada vacía
    for coord in sorted_by_x:                                                     # Recorrer lista de puntos
        list_x.append(coord)                                                      # Agregar punto a la lista ligada

    # Construir lista ligada ordenada por Y
    list_y = LinkedList()                                                         # Crear lista ligada vacía
    for coord in sorted_by_y:                                                     # Recorrer lista de puntos
        list_y.append(coord)                                                      # Agregar punto a la lista ligada

    return list_x, list_y                                                         # Retorna ambas listas ordenadas

# ------------------------------ Fuerza Bruta ------------------------------------


# -------------------------------- KD Tree ---------------------------------------

def KD_Tree(data, depth=0, k_dimension=2):                                        # Funcion para construir el KD Tree
    """
    Construye un KD-Tree a partir de una lista de puntos.

    data: lista de coordenadas (lat, lon)
    depth: profundidad actual del árbol (controla el eje de corte)
    k_dimension: número de dimensiones (2 en este caso)
    """

    if len(data) == 0:                                                            # Caso base: lista vacía
        return None

    if len(data) == 1:                                                            # Caso base: hoja
        return Node_KD_2D(data[0])

    axis = depth % k_dimension                                                    # Alterna eje: 0 → lat, 1 → lon
    other = (axis + 1) % k_dimension                                              # Eje secundario (para desempate)

    sorted_data = sorted(data, key=lambda coord: (coord[axis], coord[other]))     # Ordenar por el eje actual, usando el otro eje como criterio secundario

    mid = len(sorted_data) // 2                                                   # Índice del punto medio (mediana) (OJO: SOLO DATOS HOMOGENEOS, para distribuciones no homogeneas se implementa otro metodo)

    node = Node_KD_2D(sorted_data[mid])                                           # Nodo raíz en esta subdivisión

    # Construcción recursiva de subárboles
    node.left  = KD_Tree(sorted_data[:mid], depth + 1)                            # Subárbol izquierdo (menores)
    node.right = KD_Tree(sorted_data[mid + 1:], depth + 1)                        # Subárbol derecho (mayores)

    return node                                                                   # Retorna el nodo construido


def get_max_depth(node, depth=0):
    """
    Calcula la profundidad máxima del KD-Tree.
    """

    if node is None:                                                              # Si no hay nodo
        return depth - 1                                                          # Ajuste porque se pasó un nivel

    # Retorna la mayor profundidad entre ambos subárboles
    return max(get_max_depth(node.left, depth+1),
               get_max_depth(node.right, depth+1))

# -------------------------------- KD Tree ---------------------------------------

##**Prueba para la generación de listas, listas ligadas y árbol Kd**

In [6]:
print(f"Total de puntos generados: {len(data)}")
print(f"Primeros 3 puntos: {data[:3]}")

Total de puntos generados: 10000
Primeros 3 puntos: [(6.027705856378088, -75.33118552759076), (6.39176251408529, -75.46637060658125), (6.485348135646374, -75.23745546423163)]


In [7]:
list_x, list_y = F_bruta_list(data)
print("Lista por X:", list_x)
print("Lista por Y:", list_y)

Lista por X: (6.4854548817274456, -75.70998713639734) -> (6.14577383040385, -75.70985271041269) -> (6.129045894133644, -75.70985216541342) -> (6.352459532745714, -75.70981725181305) -> (6.082617677142981, -75.70980971891478) ... (10000 nodos)
Lista por Y: (5.980062984702583, -75.4327280764596) -> (5.980223568863618, -75.38654885258643) -> (5.980266610773789, -75.26017685982184) -> (5.980284229281461, -75.26407194793401) -> (5.980296941558624, -75.56249338775365) ... (10000 nodos)


In [8]:
# Verificación del árbol KD


def check_KD_Tree(node, depth=0, max_depth=6):
    if node is None or depth > max_depth:
        return
    axis_name = "X (lon)" if depth % 2 == 0 else "Y (lat)"
    indent = "  " * depth
    print(f"{indent}Nivel {depth} | eje: {axis_name} | valor: {node.value}")
    check_KD_Tree(node.left,  depth + 1, max_depth)
    check_KD_Tree(node.right, depth + 1, max_depth)

root = KD_Tree(data)
print(f"Profundidad máxima: {get_max_depth(root)}")
check_KD_Tree(root)

Profundidad máxima: 13
Nivel 0 | eje: X (lon) | valor: (6.247176495513584, -75.4731385536863)
  Nivel 1 | eje: Y (lat) | valor: (6.225400844540731, -75.46032725550289)
    Nivel 2 | eje: X (lon) | valor: (6.11549958785071, -75.6899107333652)
      Nivel 3 | eje: Y (lat) | valor: (6.054432583771077, -75.58735680025174)
        Nivel 4 | eje: X (lon) | valor: (6.04787433709752, -75.66859434828311)
          Nivel 5 | eje: Y (lat) | valor: (6.011717338440607, -75.65180656708469)
            Nivel 6 | eje: X (lon) | valor: (6.014067226321737, -75.67770245364322)
            Nivel 6 | eje: X (lon) | valor: (6.013518117194262, -75.62657476273172)
          Nivel 5 | eje: Y (lat) | valor: (6.071904602603428, -75.6489659704512)
            Nivel 6 | eje: X (lon) | valor: (6.083652774387369, -75.69517566549412)
            Nivel 6 | eje: X (lon) | valor: (6.082699152011402, -75.62787290048948)
        Nivel 4 | eje: X (lon) | valor: (6.0463666875132285, -75.50310057453196)
          Nivel 5 | e

##**Menú de funciones, pruebas, gráficos y rendimiento**

In [16]:
# ---------------------------------- Menú (ipywidgets) ---------------------------------------

import random, time, math                                                            # Importa las librerias para randomizado, medida de tiemnpo y calculos matematicos
import numpy as np                                                                   # Importa la libreria numpy para operaciones y calculos
import matplotlib.pyplot as plt                                                      # Importa la librria de pyuthon para graficas
import matplotlib.patches as patches                                                 # Importa de la libreria de graficas la herramienta de modificaciones
from matplotlib.lines import Line2D                                                  # Importa de la libreria de graficas la herramienta de lineas
import ipywidgets as widgets                                                         # Importa la libreria para widgets
from IPython.display import display, clear_output                                    # Importa de la libreria para visualizaciones estructuradas la herramienta de borrado y continuar

# ========================== HAVERSINE ==========================

def haversine(coord1, coord2):                                                       # Funcion para calcular la distancia entre dos puntos a paritr de latitud y longitud
    R = 6_371_000                                                                    # Radio de la tierra en metros
    lat1, lon1 = math.radians(coord1[0]), math.radians(coord1[1])                    # Convierte la latitud y longitud del primer punto de grads a radianes
    lat2, lon2 = math.radians(coord2[0]), math.radians(coord2[1])                    # Convierte la latitud y longitud del segundo punto de grads a radianes
    dlat, dlon = lat2 - lat1, lon2 - lon1                                            # Calcula la diferencia entre latitudes y longitudes entre los dos puntos
    a = math.sin(dlat/2)**2 + math.cos(lat1)*math.cos(lat2)*math.sin(dlon/2)**2      # Calculo para "a", la cual representa una medida de la separación angular entre los puntos
    return R * 2 * math.asin(math.sqrt(a))                                           # Calcula la distancia final y la devuelve en metros

# ========================== UTILIDADES ==========================

def extract_all_points(node):                                                             # Funcion para recorrer recursivamente el KD Tree
    if node is None:                                                                      # Si no hay nodo
        return []                                                                         # Retorna una lista vacia
    return [node.value] + extract_all_points(node.left) + extract_all_points(node.right)  # Si si hay nodo, toma el valor de ese nodo y lo mete en una lista, despues llama recursivamente la funcion para lso dos hijos

# ========================== BÚSQUEDA POR RADIO ==========================

def radio_fuerza_bruta(list_x, list_y, center, radio_m):                             # Recibe ambas listas ordenadas por lon y lat respectivamente para buscar vecinos a un radio dado
    radio_lat = radio_m / 111_320                                                    # Convierte el radio de metros a grados de latitud
    radio_lon = radio_m / (111_320 * math.cos(math.radians(center[0])))              # Convierte el radio de metros a grados de longitud según la latitud del centro

    # --- Filtro por longitud usando list_x (ordenada por lon) ---
    candidatos_x = set()                                                             # Conjunto para guardar puntos dentro del rango de longitud
    current = list_x.head                                                            # Primer nodo de la lista ordenada por longitud
    while current:                                                                   # Recorre la lista completa
        if abs(current.coord[1] - center[1]) <= radio_lon:                           # Si la longitud del punto está dentro del rango
            candidatos_x.add(tuple(current.coord))                                   # Se agrega al conjunto de candidatos por X
        current = current.next                                                       # Avanza al siguiente nodo

    # --- Filtro por latitud usando list_y (ordenada por lat) ---
    candidatos_y = set()                                                             # Conjunto para guardar puntos dentro del rango de latitud
    current = list_y.head                                                            # Primer nodo de la lista ordenada por latitud
    while current:                                                                   # Recorre la lista completa
        if abs(current.coord[0] - center[0]) <= radio_lat:                           # Si la latitud del punto está dentro del rango
            candidatos_y.add(tuple(current.coord))                                   # Se agrega al conjunto de candidatos por Y
        current = current.next                                                       # Avanza al siguiente nodo

    # --- Intersección: candidatos que pasan ambos filtros ---
    candidatos = candidatos_x & candidatos_y                                         # Solo los puntos dentro del rectángulo lon+lat

    # --- Verificación exacta con haversine solo sobre candidatos ---
    dentro, ops = [], 0                                                              # Lista para puntos dentro del radio y contador de operaciones
    for coord in candidatos:                                                         # Recorre solo los candidatos filtrados
        ops += 1                                                                     # Incrementa el contador de operaciones
        if haversine(center, coord) <= radio_m:                                      # Verifica si el punto está realmente dentro del radio circular
            dentro.append(coord)                                                     # Si está dentro, se agrega a la lista de resultados

    return dentro, ops                                                               # Retorna los puntos dentro del radio y el número de operaciones

def radio_kd_tree(node, center, radio_m, depth=0, k=2):                                                       # Funcion de busqueda de vecinos con poda según un radio dado (árbol ya construido)
    if node is None:                                                                                          # Caso base: si el nodo es nulo, no hay puntos que evaluar
        return [], 0                                                                                          # Retorna lista vacía y 0 operaciones
    ops = 1                                                                                                   # Contador de operaciones (visita al nodo actual)
    dentro = []                                                                                               # Lista de puntos que están dentro del radio
    if haversine(center, node.value) <= radio_m:                                                              # Calcula distancia entre el centro y el punto actual usando Haversine
        dentro.append(node.value)                                                                             # Si está dentro del radio, se agrega a la lista
    axis = depth % k                                                                                          # Determina el eje de corte (0: latitud, 1: longitud)
    val_n, val_q = node.value[axis], center[axis]                                                             # Obtiene valor del nodo y del punto de consulta en ese eje
    cerca, lejos = (node.left, node.right) if val_q < val_n else (node.right, node.left)                      # Define primero la rama "cercana" y luego la "lejana"
    res, o = radio_kd_tree(cerca, center, radio_m, depth+1, k)                                                # Llamada recursiva a la rama más prometedora
    dentro += res; ops += o                                                                                   # Acumula resultados y operaciones
    diff_m = abs(val_q - val_n) * (111_320 if axis == 0 else 111_320 * math.cos(math.radians(center[0])))     # Convierte la diferencia en el eje a metros (aproximación geográfica)
    if diff_m <= radio_m:                                                                                     # Si el radio alcanza a cruzar el plano de división
        res, o = radio_kd_tree(lejos, center, radio_m, depth+1, k)                                            # Se explora también la rama lejana
        dentro += res; ops += o                                                                               # Se acumulan resultados y operaciones adicionales
    return dentro, ops                                                                                        # Retorna puntos dentro del radio y total de operaciones

# ========================== VECINO MÁS CERCANO ==========================

def vecino_fuerza_bruta(list_x, list_y, query):                                      # Recibe ambas listas para aplicar filtrado por ventana adaptativa
    # --- Paso 1: obtener una distancia inicial desde el primer nodo de list_x ---
    mejor_inicial = None                                                             # Mejor candidato inicial antes del filtrado
    mejor_dist    = float('inf')                                                     # Distancia inicial infinita
    current = list_x.head                                                            # Recorre list_x para obtener un primer estimado de distancia
    while current:
        d = haversine(query, current.coord)                                          # Calcula distancia haversine al nodo actual
        if d < mejor_dist:                                                           # Si es menor que la mejor conocida
            mejor_dist    = d                                                        # Actualiza la mejor distancia
            mejor_inicial = current.coord                                            # Actualiza el mejor candidato
        current = current.next                                                       # Avanza al siguiente nodo
                                                                                     # Al terminar este recorrido ya tenemos el vecino real por fuerza bruta en list_x,
                                                                                     # pero aprovechamos ese mejor_dist como ventana para filtrar con list_y

    # --- Paso 2: convertir mejor_dist a grados para usarla como ventana ---
    ventana_lat = mejor_dist / 111_320                                               # Ventana en grados de latitud
    ventana_lon = mejor_dist / (111_320 * math.cos(math.radians(query[0])))          # Ventana en grados de longitud

    # --- Paso 3: filtrar list_y con la ventana de latitud ---
    candidatos_y = set()                                                             # Conjunto de candidatos que pasan el filtro de latitud
    current = list_y.head                                                            # Primer nodo de la lista ordenada por latitud
    while current:                                                                   # Recorre list_y completa
        if abs(current.coord[0] - query[0]) <= ventana_lat:                          # Si la latitud cae dentro de la ventana
            candidatos_y.add(tuple(current.coord))                                   # Se agrega como candidato
        current = current.next                                                       # Avanza al siguiente nodo

    # --- Paso 4: filtrar list_x con la ventana de longitud e intersectar ---
    candidatos_x = set()                                                             # Conjunto de candidatos que pasan el filtro de longitud
    current = list_x.head                                                            # Primer nodo de la lista ordenada por longitud
    while current:                                                                   # Recorre list_x completa
        if abs(current.coord[1] - query[1]) <= ventana_lon:                          # Si la longitud cae dentro de la ventana
            candidatos_x.add(tuple(current.coord))                                   # Se agrega como candidato
        current = current.next                                                       # Avanza al siguiente nodo
    candidatos = candidatos_x & candidatos_y                                         # Intersección: puntos dentro del rectángulo lat+lon

    # --- Paso 5: haversine exacto solo sobre los candidatos de la intersección ---
    mejor, ops = mejor_inicial, 0                                                    # Parte del mejor ya encontrado en el paso 1
    for coord in candidatos:                                                         # Recorre solo los candidatos filtrados
        ops += 1                                                                     # Incrementa el contador de operaciones
        d = haversine(query, coord)                                                  # Calcula distancia exacta
        if d < mejor_dist:                                                           # Si supera al mejor actual
            mejor_dist = d                                                           # Actualiza la mejor distancia
            mejor      = coord                                                       # Actualiza el mejor candidato
    return mejor, mejor_dist, ops                                                    # Retorna el vecino más cercano, su distancia y operaciones realizadas


def vecino_kd_tree(node, query, depth=0, k=2, mejor=None, mejor_dist=float('inf'), ops=0):                # Búsqueda del vecino más cercano en un KD-Tree con poda
    if node is None:                                                                                      # Caso base: si el nodo es nulo, no hay más por explorar
        return mejor, mejor_dist, ops                                                                     # Retorna el mejor encontrado hasta ahora

    ops += 1                                                                                              # Cuenta la visita al nodo actual
    d = haversine(query, node.value)                                                                      # Calcula la distancia entre el punto de consulta y el nodo actual
    if d < mejor_dist:                                                                                    # Si la distancia es menor que la mejor conocida
        mejor_dist, mejor = d, node.value                                                                 # Actualiza el mejor punto y su distancia
    axis = depth % k                                                                                      # Determina el eje de corte (0: lat, 1: lon)
    val_n, val_q = node.value[axis], query[axis]                                                          # Valores del nodo y del punto de consulta en ese eje
    cerca, lejos = (node.left, node.right) if val_q < val_n else (node.right, node.left)                  # Define la rama cercana y la lejana
    mejor, mejor_dist, ops = vecino_kd_tree(cerca, query, depth+1, k, mejor, mejor_dist, ops)             # Explora primero la rama más prometedora
    diff_m = abs(val_q - val_n) * (111_320 if axis == 0 else 111_320 * math.cos(math.radians(query[0])))  # Distancia al plano de corte en metros (aprox.)
    if diff_m < mejor_dist:                                                                               # Si existe posibilidad de encontrar un punto más cercano en la otra rama
        mejor, mejor_dist, ops = vecino_kd_tree(lejos, query, depth+1, k, mejor, mejor_dist, ops)         # Explora la rama lejana

    return mejor, mejor_dist, ops                                                                         # Retorna el vecino más cercano, su distancia y el número de operaciones

# ========================== GRÁFICOS ==========================

def graficar_radio(todos, dentro, center, radio_m, titulo=""):
    lons_t = [p[1] for p in todos]; lats_t = [p[0] for p in todos]
    lons_d = [p[1] for p in dentro]; lats_d = [p[0] for p in dentro]
    fuera  = [p for p in todos if p not in set(map(tuple, dentro))]
    lons_f = [p[1] for p in fuera]; lats_f = [p[0] for p in fuera]
    radio_lat = radio_m / 111_320
    radio_lon = radio_m / (111_320 * math.cos(math.radians(center[0])))

    fig, axes = plt.subplots(1, 2, figsize=(16, 6))
    fig.suptitle(titulo, fontsize=13, fontweight='bold')
    for ax, zoom in zip(axes, [False, True]):
        ax.scatter(lons_f, lats_f, c='steelblue', s=15, alpha=0.5, label='Fuera del radio', zorder=2)
        if lons_d:
            ax.scatter(lons_d, lats_d, c='darkorange', s=45, alpha=0.9, label='Dentro del radio', zorder=3)
        ax.scatter(center[1], center[0], c='red', s=90, marker='*', label='Centro', zorder=5)
        circ = patches.Ellipse((center[1], center[0]), width=2*radio_lon, height=2*radio_lat,
                                lw=1.5, edgecolor='red', facecolor='salmon', alpha=0.15, zorder=4)
        ax.add_patch(circ)
        if zoom:
            m = 2.0
            ax.set_xlim(center[1]-radio_lon*m, center[1]+radio_lon*m)
            ax.set_ylim(center[0]-radio_lat*m, center[0]+radio_lat*m)
            ax.set_title("Zoom al radio")
        else:
            ax.set_title("Mapa completo")
        ax.set_xlabel("Longitud"); ax.set_ylabel("Latitud")
        ax.legend(fontsize=8); ax.grid(True, alpha=0.3)
    plt.tight_layout(); plt.show()

def graficar_vecino(puntos_entrega, query, vecino, dist_m):
    lons_e = [p[1] for p in puntos_entrega]; lats_e = [p[0] for p in puntos_entrega]
    fig, ax = plt.subplots(figsize=(9, 6))
    ax.scatter(lons_e, lats_e, c='steelblue', s=25, alpha=0.6, label='Puntos de entrega', zorder=2)
    ax.scatter(query[1], query[0], c='red', s=110, marker='*', label='Consulta', zorder=5)
    if vecino:
        ax.scatter(vecino[1], vecino[0], c='darkorange', s=90, marker='D',
                   label=f'Más cercano ({dist_m:.1f} m)', zorder=4)
        ax.plot([query[1], vecino[1]], [query[0], vecino[0]], 'k--', lw=1.2, alpha=0.6, zorder=3)
    ax.set_title("Vecino más cercano entre puntos de entrega", fontsize=12, fontweight='bold')
    ax.set_xlabel("Longitud"); ax.set_ylabel("Latitud")
    ax.legend(fontsize=9); ax.grid(True, alpha=0.3)
    plt.tight_layout(); plt.show()

def graficar_kd_tree(node, todos, profundidad_max=13):
    lats = [p[0] for p in todos]; lons = [p[1] for p in todos]
    colores = ['#e74c3c','#3498db','#2ecc71','#f39c12','#9b59b6','#1abc9c']
    fig, ax = plt.subplots(figsize=(10, 7))
    ax.scatter(lons, lats, c='gray', s=10, alpha=0.4, zorder=2)
    ax.set_title("Particiones del KD-Tree (por nivel)", fontsize=13, fontweight='bold')
    ax.set_xlabel("Longitud"); ax.set_ylabel("Latitud")

    def dibujar(n, x0, x1, y0, y1, d):
        if n is None or d > profundidad_max: return
        col = colores[d % len(colores)]
        lw  = max(0.4, 1.8 - d*0.25)
        lat_n, lon_n = n.value
        if d % 2 == 0:
            # axis=0 → latitud → línea HORIZONTAL
            ax.plot([x0, x1], [lat_n, lat_n], color=col, lw=lw, alpha=0.75, zorder=3)
            dibujar(n.left,  x0, x1, y0, lat_n, d+1)
            dibujar(n.right, x0, x1, lat_n, y1, d+1)
        else:
            # axis=1 → longitud → línea VERTICAL
            ax.plot([lon_n, lon_n], [y0, y1], color=col, lw=lw, alpha=0.75, zorder=3)
            dibujar(n.left,  x0, lon_n, y0, y1, d+1)
            dibujar(n.right, lon_n, x1, y0, y1, d+1)
        ax.scatter(lon_n, lat_n, c=col, s=22, zorder=4, edgecolors='white', lw=0.5)

    dibujar(node, min(lons), max(lons), min(lats), max(lats), 0)
    leyenda = [Line2D([0],[0], color=colores[i%len(colores)], lw=1.5,
                      label=f'Nivel {i} ({"lat" if i%2==0 else "lon"})') for i in range(profundidad_max+1)]
    ax.legend(handles=leyenda, fontsize=7, loc='lower right', ncol=2)
    ax.grid(True, alpha=0.25)
    plt.tight_layout(); plt.show()

def imprimir_tabla(fb_t, kd_t, fb_ops, kd_ops, fb_res, kd_res, nombre):
    speedup = fb_t / kd_t if kd_t > 0 else float('inf')
    iguales = set(map(tuple, fb_res)) == set(map(tuple, kd_res))
    print(f"\n{'='*56}")
    print(f"  Comparación: {nombre}")
    print(f"{'='*56}")
    print(f"  {'Métrica':<26} {'Lista Ligada':>13} {'KD-Tree':>13}")
    print(f"  {'-'*52}")
    print(f"  {'Tiempo (ms)':<26} {fb_t*1000:>13.4f} {kd_t*1000:>13.4f}")
    print(f"  {'Nodos visitados':<26} {fb_ops:>13} {kd_ops:>13}")
    print(f"  {'Speedup KD vs FB':<26} {speedup:>13.2f}x")
    print(f"  {'Complejidad teórica':<26} {'O(n)':>13} {'O(log n)':>13}")
    print(f"  {'Resultado idéntico':<26} {'Sí' if iguales else 'No':>13}")
    print(f"{'='*56}\n")

# ========================== ZOOM KD-TREE ==========================

def graficar_kd_tree_zoom(todos, n_puntos=25, profundidad_max=5):
    muestra = random.sample(todos, min(n_puntos, len(todos)))
    arbol_zoom = KD_Tree(muestra)

    lats = [p[0] for p in muestra]
    lons = [p[1] for p in muestra]
    colores = ['#e74c3c','#3498db','#2ecc71','#f39c12','#9b59b6','#1abc9c']

    fig, ax = plt.subplots(figsize=(10, 7))
    ax.set_title(f"Zoom KD-Tree — {len(muestra)} puntos (cortes por nodo)", fontsize=13, fontweight='bold')
    ax.set_xlabel("Longitud"); ax.set_ylabel("Latitud")
    ax.grid(True, alpha=0.3)

    margen_lat = (max(lats) - min(lats)) * 0.05
    margen_lon = (max(lons) - min(lons)) * 0.05
    ax.set_xlim(min(lons) - margen_lon, max(lons) + margen_lon)
    ax.set_ylim(min(lats) - margen_lat, max(lats) + margen_lat)

    def dibujar(n, x0, x1, y0, y1, d):
        if n is None or d > profundidad_max: return
        col = colores[d % len(colores)]
        lw  = max(0.6, 2.2 - d * 0.3)
        lat_n, lon_n = n.value

        if d % 2 == 0:
            # axis=0 → latitud → línea HORIZONTAL pasando por el nodo
            ax.plot([x0, x1], [lat_n, lat_n], color=col, lw=lw, alpha=0.85, zorder=3)
            ax.annotate(f"L{d}", (lon_n, lat_n),
                        textcoords="offset points", xytext=(5, 5),
                        fontsize=7, color=col, fontweight='bold')
            dibujar(n.left,  x0, x1, y0, lat_n, d+1)
            dibujar(n.right, x0, x1, lat_n, y1,  d+1)
        else:
            # axis=1 → longitud → línea VERTICAL pasando por el nodo
            ax.plot([lon_n, lon_n], [y0, y1], color=col, lw=lw, alpha=0.85, zorder=3)
            ax.annotate(f"L{d}", (lon_n, lat_n),
                        textcoords="offset points", xytext=(5, 5),
                        fontsize=7, color=col, fontweight='bold')
            dibujar(n.left,  x0, lon_n, y0, y1, d+1)
            dibujar(n.right, lon_n, x1, y0, y1, d+1)

        ax.scatter(lon_n, lat_n, c=col, s=60, zorder=5, edgecolors='white', lw=1.0)

    dibujar(arbol_zoom, min(lons) - margen_lon, max(lons) + margen_lon,
                        min(lats) - margen_lat, max(lats) + margen_lat, 0)

    ax.scatter(lons, lats, c='gray', s=20, alpha=0.3, zorder=2)

    leyenda = [Line2D([0],[0], color=colores[i % len(colores)], lw=1.8,
                      label=f'Nivel {i} ({"lat→horiz" if i%2==0 else "lon→vert"})')
               for i in range(profundidad_max + 1)]
    ax.legend(handles=leyenda, fontsize=8, loc='lower right')
    plt.tight_layout(); plt.show()

# ========================== RENDIMIENTO ==========================

def prueba_rendimiento(todos, list_x, list_y, arbol, n_iter=100):
    min_lat = min(p[0] for p in todos); max_lat = max(p[0] for p in todos)                                              # Calcula límites de latitud del conjunto de datos
    min_lon = min(p[1] for p in todos); max_lon = max(p[1] for p in todos)                                              # Calcula límites de longitud

    # --- Radio ---
    t_fb_radio, t_kd_radio = [], []                                                                                     # Listas para tiempos (fuerza bruta vs KD-Tree)
    ops_fb_radio, ops_kd_radio = [], []                                                                                 # Listas para número de operaciones

    for _ in range(n_iter):                                                                                             # Repite el experimento varias veces
        center  = (random.uniform(min_lat, max_lat), random.uniform(min_lon, max_lon))                                  # Genera punto aleatorio dentro del área
        radio_m = random.uniform(200, 1000)                                                                             # Genera un radio aleatorio entre 200m y 1000m
        t0 = time.perf_counter(); _, ops = radio_fuerza_bruta(list_x, list_y, center, radio_m); t_fb_radio.append(time.perf_counter()-t0); ops_fb_radio.append(ops)
        # Ejecuta búsqueda por radio con fuerza bruta, mide tiempo y operaciones
        t0 = time.perf_counter(); _, ops = radio_kd_tree(arbol, center, radio_m);               t_kd_radio.append(time.perf_counter()-t0); ops_kd_radio.append(ops)
        # Ejecuta búsqueda por radio con KD-Tree, mide tiempo y operaciones

    # --- Vecino ---
    t_fb_vec, t_kd_vec = [], []                                                                                         # Listas de tiempos para vecino más cercano
    ops_fb_vec, ops_kd_vec = [], []                                                                                     # Listas de operaciones

    for _ in range(n_iter):                                                                                             # Repite experimento
        n_e       = max(1, int(len(todos) * 0.05))                                                                      # Toma un subconjunto (~5%) de los datos
        entrega   = random.sample(todos, n_e)                                                                           # Selecciona puntos aleatorios
        list_e_x, list_e_y = F_bruta_list(entrega)                                                                      # Convierte a estructura para fuerza bruta
        arbol_e   = KD_Tree(entrega)                                                                                    # Construye KD-Tree para ese subconjunto
        query     = (random.uniform(min_lat, max_lat), random.uniform(min_lon, max_lon))                                # Punto de consulta aleatorio
        t0 = time.perf_counter(); _, _, ops = vecino_fuerza_bruta(list_e_x, list_e_y, query); t_fb_vec.append(time.perf_counter()-t0); ops_fb_vec.append(ops)
        # Búsqueda de vecino más cercano por fuerza bruta
        t0 = time.perf_counter(); _, _, ops = vecino_kd_tree(arbol_e, query); t_kd_vec.append(time.perf_counter()-t0); ops_kd_vec.append(ops)
        # Búsqueda de vecino más cercano usando KD-Tree

    # --- Imprimir resultados ---
    def bloque(nombre, fb_t, kd_t, fb_ops, kd_ops):
        speedup = np.mean(fb_t) / np.mean(kd_t) if np.mean(kd_t) > 0 else float('inf')
        print(f"\n{'='*62}")
        print(f"  {nombre}  ({n_iter} iteraciones)")
        print(f"{'='*62}")
        print(f"  {'Métrica':<28} {'Lista Ligada':>15} {'KD-Tree':>15}")
        print(f"  {'-'*58}")
        print(f"  {'Tiempo medio (ms)':<28} {np.mean(fb_t)*1000:>15.4f} {np.mean(kd_t)*1000:>15.4f}")
        print(f"  {'Tiempo mínimo (ms)':<28} {np.min(fb_t)*1000:>15.4f} {np.min(kd_t)*1000:>15.4f}")
        print(f"  {'Tiempo máximo (ms)':<28} {np.max(fb_t)*1000:>15.4f} {np.max(kd_t)*1000:>15.4f}")
        print(f"  {'Desv. estándar (ms)':<28} {np.std(fb_t)*1000:>15.4f} {np.std(kd_t)*1000:>15.4f}")
        print(f"  {'Nodos visitados (medio)':<28} {np.mean(fb_ops):>15.1f} {np.mean(kd_ops):>15.1f}")
        print(f"  {'Speedup KD vs FB':<28} {speedup:>15.2f}x")
        print(f"{'='*62}")

    bloque("Búsqueda por radio", t_fb_radio, t_kd_radio, ops_fb_radio, ops_kd_radio)
    bloque("Vecino más cercano", t_fb_vec,   t_kd_vec,   ops_fb_vec,   ops_kd_vec)

    # --- Gráfico comparativo ---
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    fig.suptitle(f"Prueba de rendimiento — {n_iter} iteraciones", fontsize=13, fontweight='bold')

    for ax, fb_t, kd_t, nombre in zip(
        axes,
        [t_fb_radio, t_fb_vec],
        [t_kd_radio, t_kd_vec],
        ["Búsqueda por radio", "Vecino más cercano"]
    ):
        iters = range(1, n_iter + 1)
        ax.plot(iters, [t*1000 for t in fb_t], color='steelblue', lw=0.8, alpha=0.6, label='Lista ligada')
        ax.plot(iters, [t*1000 for t in kd_t], color='darkorange', lw=0.8, alpha=0.6, label='KD-Tree')
        ax.axhline(np.mean(fb_t)*1000, color='steelblue', lw=1.5, ls='--', alpha=0.9)
        ax.axhline(np.mean(kd_t)*1000, color='darkorange', lw=1.5, ls='--', alpha=0.9)
        ax.set_title(nombre); ax.set_xlabel("Iteración"); ax.set_ylabel("Tiempo (ms)")
        ax.legend(fontsize=9); ax.grid(True, alpha=0.3)

    plt.tight_layout(); plt.show()

# ========================== CONSTRUCCIÓN DE ESTRUCTURAS ==========================

todos          = extract_all_points(root)    # 'root' debe estar ya construido
list_x, list_y = F_bruta_list(todos)         # Guarda ambas listas, ordenadas por lon y lat
arbol          = root

min_lat, max_lat = min(p[0] for p in todos), max(p[0] for p in todos)
min_lon, max_lon = min(p[1] for p in todos), max(p[1] for p in todos)

# ========================== WIDGETS ==========================

out = widgets.Output()

btn_radio      = widgets.Button(description="Búsqueda por radio (500 m)",
                                button_style='primary', layout=widgets.Layout(width='260px'))
btn_vecino     = widgets.Button(description="Vecino más cercano",
                                button_style='info',    layout=widgets.Layout(width='260px'))
btn_arbol      = widgets.Button(description="Visualizar KD-Tree",
                                button_style='warning', layout=widgets.Layout(width='260px'))
btn_zoom       = widgets.Button(description="Zoom KD-Tree (25 pts)",
                                button_style='success', layout=widgets.Layout(width='220px'))
btn_rendimiento = widgets.Button(description="Prueba de rendimiento",
                                 button_style='danger',  layout=widgets.Layout(width='220px'))

header = widgets.HTML(
    f"<h3 style='margin:0 0 4px'>Sistema de puntos de entrega</h3>"
    f"<p style='color:gray;margin:0'>KD-Tree vs Lista Ligada &nbsp;|&nbsp; {len(todos)} puntos cargados</p>"
)

panel = widgets.VBox([
    header,
    widgets.HBox([btn_radio, btn_vecino, btn_arbol, btn_zoom, btn_rendimiento],
                 layout=widgets.Layout(gap='10px', margin='12px 0 0', flex_wrap='wrap')),
    out
])

# ---- Manejadores ----

def on_radio(b):                                                                                       # Evento: ejecuta búsqueda por radio al presionar botón
    with out:                                                                                          # Redirige la salida al widget 'out'
        clear_output(wait=True)                                                                        # Limpia la salida anterior para actualizar la vista
        center = (random.uniform(min_lat, max_lat), random.uniform(min_lon, max_lon))                  # Genera un punto central aleatorio
        radio_m = 500                                                                                  # Define un radio fijo de 500 metros
        print(f"Centro aleatorio: lat={center[0]:.6f}, lon={center[1]:.6f}")                           # Muestra coordenadas del centro
        print(f"Radio: {radio_m} m\n")                                                                 # Muestra el radio usado
        t0 = time.perf_counter(); fb_res, fb_ops = radio_fuerza_bruta(list_x, list_y, center, radio_m)
        fb_t = time.perf_counter()-t0                                                                  # Ejecuta fuerza bruta y mide tiempo
        t0 = time.perf_counter(); kd_res, kd_ops = radio_kd_tree(arbol, center, radio_m)
        kd_t = time.perf_counter()-t0                                                                  # Ejecuta KD-Tree y mide tiempo
        print(f"Puntos encontrados  →  Lista ligada: {len(fb_res)}  |  KD-Tree: {len(kd_res)}")        # Compara resultados
        imprimir_tabla(fb_t, kd_t, fb_ops, kd_ops, fb_res, kd_res, "Búsqueda por radio 500 m")         # Muestra tabla comparativa
        graficar_radio(todos, kd_res, center, radio_m,                                                 # Grafica resultados usando KD-Tree
                       titulo=f"Radio 500 m — centro ({center[0]:.5f}, {center[1]:.5f})")

def on_vecino(b):                                                                                      # Evento: ejecuta búsqueda de vecino más cercano
    with out:
        clear_output(wait=True)                                                                        # Limpia salida
        n_e     = max(1, int(len(todos) * 0.05))                                                       # Selecciona ~5% de los puntos
        entrega = random.sample(todos, n_e)                                                            # Subconjunto aleatorio
        list_e_x, list_e_y = F_bruta_list(entrega)                                                     # Estructura para fuerza bruta
        arbol_e = KD_Tree(entrega)                                                                     # Construye KD-Tree del subconjunto
        query   = (random.uniform(min_lat, max_lat), random.uniform(min_lon, max_lon))                 # Punto de consulta aleatorio
        print(f"Puntos de entrega (5% aleatorio): {n_e}")                                              # Muestra tamaño del subconjunto
        print(f"Consulta: lat={query[0]:.6f}, lon={query[1]:.6f}\n")                                   # Muestra punto de consulta
        t0 = time.perf_counter(); fb_v, fb_d, fb_ops = vecino_fuerza_bruta(list_e_x, list_e_y, query)
        fb_t = time.perf_counter()-t0                                                                  # Fuerza bruta
        t0 = time.perf_counter(); kd_v, kd_d, kd_ops = vecino_kd_tree(arbol_e, query)
        kd_t = time.perf_counter()-t0                                                                  # KD-Tree
        print(f"Lista ligada → {fb_v}  ({fb_d:.2f} m)")                                                # Resultado fuerza bruta
        print(f"KD-Tree      → {kd_v}  ({kd_d:.2f} m)")                                                # Resultado KD-Tree
        imprimir_tabla(fb_t, kd_t, fb_ops, kd_ops,                                                     # Tabla comparativa
                       [fb_v] if fb_v else [], [kd_v] if kd_v else [],
                       "Vecino más cercano")
        graficar_vecino(entrega, query, kd_v, kd_d)                                                    # Grafica el vecino más cercano

def on_arbol(b):                                                                                       # Evento: visualizar particiones del KD-Tree
    with out:
        clear_output(wait=True)
        print("Graficando particiones del KD-Tree...\n")                                               # Mensaje informativo
        graficar_kd_tree(arbol, todos, profundidad_max=5)                                              # Grafica divisiones del árbol

def on_zoom(b):                                                                                        # Evento: zoom sobre una pequeña muestra
    with out:
        clear_output(wait=True)
        print("Generando zoom con 25 puntos aleatorios...\n")                                          # Mensaje informativo
        graficar_kd_tree_zoom(todos, n_puntos=25, profundidad_max=4)                                   # Grafica zoom del KD-Tree

def on_rendimiento(b):                                                                                 # Evento: ejecutar pruebas de rendimiento
    with out:
        clear_output(wait=True)
        print("Corriendo iteraciones... esto puede tardar unos segundos.\n")                           # Aviso al usuario
        prueba_rendimiento(todos, list_x, list_y, arbol, n_iter=100)                                   # Ejecuta benchmark completo

btn_radio.on_click(on_radio)
btn_vecino.on_click(on_vecino)
btn_arbol.on_click(on_arbol)
btn_zoom.on_click(on_zoom)
btn_rendimiento.on_click(on_rendimiento)

display(panel)